In [7]:
from pathlib import Path

import pandas as pd
from nltk import Tree

PROJECT_ROOT = Path("..").resolve()
df = pd.read_feather(
    PROJECT_ROOT / "data" / "processed" / "constituency_parses.feather"
)

print(df.shape)
print(df.dtypes)
df.head()

(253049, 6)
doc_id         str
domain         str
source         str
sent_idx     int64
sent_text      str
parse_str      str
dtype: object


,doc_id,domain,source,sent_idx,sent_text,parse_str
0,1,essay,human,0,The 2013 film 12 Years a Slave proved that sla...,(S (NP (NP (DT The) (CD 2013) (NN film)) (NP (...
1,1,essay,human,1,"Indeed, the film made $150 million outside the...","(S (ADVP (RB Indeed)) (, ,) (NP (DT the) (NN f..."
2,1,essay,human,2,The movie was based on the memoir Twelve Years...,(S (NP (DT The) (NN movie)) (VP (VBD was) (VP ...
3,1,essay,human,3,It tells the story of a free African American ...,(S (NP (PRP It)) (VP (VBZ tells) (NP (NP (DT t...
4,1,essay,human,4,Solomon spent twelve years away from his famil...,(S (NP (NNP Solomon)) (VP (VBD spent) (NP (CD ...


In [8]:
# Convert parse string to tree for first row
parse_string = df.iloc[0]['parse_str']
tree = Tree.fromstring(parse_string)
print(tree.pretty_print())

                                                S                                                 
                ________________________________|_______________________________________________   
               |                                      VP                                        | 
               |                                 _____|____________                             |  
               |                                |                 SBAR                          | 
               |                                |      ____________|____                        |  
               NP                               |     |                 S                       | 
      _________|_____________                   |     |       __________|___                    |  
     |                       NP                 |     |      |              VP                  | 
     |              _________|_______           |     |      |      ________|______             |  
     

In [9]:
def feng_algo_1(tree: Tree) -> str:
    # sourcery skip: assign-if-exp, merge-else-if-into-elif, remove-unnecessary-else, swap-if-else-branches
    l_top: list[str] = [c.label() for c in tree if isinstance(c, Tree)]
    all_labels: set[str] = {s.label() for s in tree.subtrees()}
    if 'S' in l_top:
        if 'SBAR' not in all_labels:
            return 'COMPOUND'
        else:
            return 'COMPLEX-COMPOUND'
    else:
        if 'VP' in l_top:
            if 'SBAR' not in all_labels:
                return 'SIMPLE'
            else:
                return 'COMPLEX'
    return 'OTHER'

In [10]:
print(feng_algo_1(tree))

COMPLEX


In [11]:
def feng_algo_2(tree: Tree) -> str:
    # sourcery skip: merge-else-if-into-elif, swap-if-else-branches, while-to-for
    l_top = [c for c in tree if isinstance(c, Tree)]
    lam = len(l_top)
    k = 0
    while k < lam:
        node = l_top[k]
        labels_in_subtree = {s.label() for s in node.subtrees()}
        
        if node.label() != 'VP':
            if 'S' in labels_in_subtree or 'SBAR' in labels_in_subtree:
                return 'PERIODIC'
        else:
            if 'S' in labels_in_subtree or 'SBAR' in labels_in_subtree:
                return 'LOOSE'
        k += 1
    return 'OTHER'

In [12]:
print(feng_algo_2(tree))

LOOSE


In [13]:
# Add 2 new columns to df with the results of feng_algo_1 and feng_algo_2
df['feng_algo_1'] = df['parse_str'].apply(lambda x: feng_algo_1(Tree.fromstring(x)))
df['feng_algo_2'] = df['parse_str'].apply(lambda x: feng_algo_2(Tree.fromstring(x)))

In [14]:
print(df.head())

  doc_id domain source  sent_idx  \
0      1  essay  human         0   
1      1  essay  human         1   
2      1  essay  human         2   
3      1  essay  human         3   
4      1  essay  human         4   

                                           sent_text  \
0  The 2013 film 12 Years a Slave proved that sla...   
1  Indeed, the film made $150 million outside the...   
2  The movie was based on the memoir Twelve Years...   
3  It tells the story of a free African American ...   
4  Solomon spent twelve years away from his famil...   

                                           parse_str feng_algo_1 feng_algo_2  
0  (S (NP (NP (DT The) (CD 2013) (NN film)) (NP (...     COMPLEX       LOOSE  
1  (S (ADVP (RB Indeed)) (, ,) (NP (DT the) (NN f...      SIMPLE       OTHER  
2  (S (NP (DT The) (NN movie)) (VP (VBD was) (VP ...      SIMPLE       OTHER  
3  (S (NP (PRP It)) (VP (VBZ tells) (NP (NP (DT t...     COMPLEX       LOOSE  
4  (S (NP (NNP Solomon)) (VP (VBD spent) (NP (CD ..

In [16]:
completed_df = pd.read_feather(PROJECT_ROOT / "data" / "processed" / "constituency_features.feather")
completed_df.head()

,doc_id,domain,source,sent_idx,sent_text,parse_str,feng_algo_1,feng_algo_2
0,1,essay,human,0,The 2013 film 12 Years a Slave proved that sla...,(S (NP (NP (DT The) (CD 2013) (NN film)) (NP (...,COMPLEX,LOOSE
1,1,essay,human,1,"Indeed, the film made $150 million outside the...","(S (ADVP (RB Indeed)) (, ,) (NP (DT the) (NN f...",SIMPLE,OTHER
2,1,essay,human,2,The movie was based on the memoir Twelve Years...,(S (NP (DT The) (NN movie)) (VP (VBD was) (VP ...,SIMPLE,OTHER
3,1,essay,human,3,It tells the story of a free African American ...,(S (NP (PRP It)) (VP (VBZ tells) (NP (NP (DT t...,COMPLEX,LOOSE
4,1,essay,human,4,Solomon spent twelve years away from his famil...,(S (NP (NNP Solomon)) (VP (VBD spent) (NP (CD ...,SIMPLE,LOOSE
